# Parsing Real Documents & Chunking Strategies

### 1) Objective
To demonstrate how to parse various real-world document formats (PDF, HTML, DOCX) and apply different chunking strategies to optimize text for Retrieval-Augmented Generation (RAG) systems.

### 2) Libraries Used
- **PyMuPDF (`fitz`)**: For fast, standard text extraction from PDFs.
- **pdfplumber**: For precise extraction of tabular data from PDFs.
- **BeautifulSoup4**: For navigating and cleaning HTML document trees.
- **python-docx**: For extracting structured text and heading styles from Word documents.
- **ollama & numpy**: For local embedding generation and semantic similarity calculations.

### 3) Ambition of the Code
Provide a robust foundational pipeline for ingesting messy, real-world corporate documents (like KinkyShoeBros policies and catalogs), extracting both structured and unstructured data, and intelligently slicing it to preserve semantic meaning for downstream AI applications.

---

### Understanding the Code & Concepts

**1) What the code expects to do:**
The code loads sample KinkyShoeBros documents, extracts their raw text and tables using format-specific parsers, handles edge cases (like noisy navbars or scanned PDFs), and slices the text into chunks using four different methods (fixed, recursive, semantic, and structure-aware).

**2) What the outputs mean:**
The outputs show the extracted text snippets and how they are divided. A "good" output chunk contains a complete, self-contained thought without cutting off mid-sentence or losing its structural context (like section headers). Poor outputs show mid-sentence breaks or jumbled table data.

**3) What it means to Chunking:**
Chunking is the process of breaking down large documents into smaller, manageable pieces (chunks). This ensures that when an AI searches for information, it retrieves precise, relevant context rather than overwhelming the system with an entire document.

**4) What it means to Parsing:**
Parsing is the mechanical extraction of raw text and structural elements (like tables, headers, or paragraphs) from native file formats (like PDF or HTML) so that a machine can read, structure, and process the data.

---

## Cell 1 — Setup

Import parsers and confirm the sample documents are present. If any are
missing, run your document generation script first.

In [1]:
from pathlib import Path

SAMPLE_DIR = Path.cwd() / "sample_docs"

SAMPLE_PDF  = SAMPLE_DIR / "kinkyshoebros_policies.pdf"
SAMPLE_HTML = SAMPLE_DIR / "kinkyshoebros_catalog.html"
SAMPLE_DOCX = SAMPLE_DIR / "kinkyshoebros_preloved.docx"

# Hard-fail if any are missing — the notebook depends on them
for path in [SAMPLE_PDF, SAMPLE_HTML, SAMPLE_DOCX]:
    assert path.exists(), (
        f"Missing {path.name}. Please generate the documents first."
    )

print("Sample documents ready:")
for path in [SAMPLE_PDF, SAMPLE_HTML, SAMPLE_DOCX]:
    print(f"  {path.name:30s}  {path.stat().st_size:>6d} bytes")

Sample documents ready:
  kinkyshoebros_policies.pdf        3684 bytes
  kinkyshoebros_catalog.html        3215 bytes
  kinkyshoebros_preloved.docx      37703 bytes


---

## Cell 2 — PyMuPDF on the sample PDF

**PyMuPDF (imported as `fitz` or `pymupdf`)** is the primary PDF library.
Fast, good text extraction, handles most PDFs.

Extract text page-by-page and observe what comes back.

In [2]:
import pymupdf  # aka fitz

doc = pymupdf.open(SAMPLE_PDF)
print(f"PDF: {SAMPLE_PDF.name}")
print(f"Pages: {len(doc)}\n")

for page_num, page in enumerate(doc, 1):
    text = page.get_text()
    print(f"══ Page {page_num} ({len(text)} chars) ══")
    # Show first 250 chars of each page
    preview = text[:250].replace('\n', ' | ')
    print(f"{preview}...\n")

doc.close()

PDF: kinkyshoebros_policies.pdf
Pages: 2

══ Page 1 (1238 chars) ══
kinkyShoeBros — Internal Employee Handbook | Section 1: Employee Footwear Allocation Program | As representatives of kinkyShoeBros, retail and corporate employees are required to wear approved | footwear during operating hours. The company provides a quart...

══ Page 2 (264 chars) ══
Section 2: HR & Support Contacts | For allocation adjustments, contact HR Business Partner Sarah Chen at | sarah.chen@kinkyshoebros.com (+1-555-0142). For payroll or stipend questions, contact Michael | Torres in Finance (EMP-04521) at michael.torres@kinky...



**Discussion moment:**
- Notice PyMuPDF preserves paragraph breaks (as `\n\n`) — this helps chunking later
- Look at page 1 — the table appears as jumbled columns. **PyMuPDF struggles with tables.**
- For pure paragraph text, PyMuPDF is fine. For tables, we need something better — next cell.

---

## Cell 3 — pdfplumber on the same PDF

**pdfplumber** is slower than PyMuPDF but much better at tables. Use it when
your corpus has structured tabular data.

Same file, watch the difference on the shoe allocation table.

In [3]:
%pip install pdfplumber

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pdfplumber

with pdfplumber.open(SAMPLE_PDF) as pdf:
    first_page = pdf.pages[0]
    
    # Text extraction — similar to PyMuPDF for prose
    text = first_page.extract_text()
    print("Text extraction (same as PyMuPDF for prose):")
    print(text[:300], "...\n")
    
    # Table extraction — pdfplumber's superpower
    tables = first_page.extract_tables()
    print(f"Tables detected on page 1: {len(tables)}")
    for i, table in enumerate(tables, 1):
        print(f"\n══ Table {i} ══")
        for row in table:
            print(f"  {' | '.join(str(c) for c in row)}")

Text extraction (same as PyMuPDF for prose):
kinkyShoeBros — Internal Employee Handbook
Section 1: Employee Footwear Allocation Program
As representatives of kinkyShoeBros, retail and corporate employees are required to wear approved
footwear during operating hours. The company provides a quarterly stipend and direct allocations to
ensure staf ...

Tables detected on page 1: 1

══ Table 1 ══
  Department / Role | Approved Brand | Model | Allocation Frequency
  Retail Store Manager | Nike | Air Max 95 | Quarterly
  Retail Assistant Mgr | adidas | Ultraboost 1.0 | Quarterly
  Sales Associate Level 3 | New Balance | 990v6 | Bi-Annually
  Sales Associate Level 2 | ASICS | Gel-Kayano 30 | Bi-Annually
  Sales Associate Level 1 | Nike | Air Force 1 | Annually
  Stockroom Manager | PUMA | Suede Classic | Bi-Annually
  Stockroom Assistant | adidas | Stan Smith | Annually
  Warehouse Operations | ASICS | Gel-Nimbus | Bi-Annually
  Logistics Coordinator | New Balance | 2002R | Bi-Annually
  Marke

**Discussion moment:**
- pdfplumber returns the table as a list-of-lists — proper structure preserved
- For RAG, this matters: a chunk of `'Retail Store Manager | Nike | Air Max 95'` is useful; a jumbled
  string might lose column semantics
- **Rule of thumb:** try PyMuPDF first (fast). If your corpus has tables, add pdfplumber
  as a fallback and merge the outputs.

---

## Cell 4 — BeautifulSoup on the HTML page

**BeautifulSoup** parses HTML into a navigable tree. For RAG, the important
operation is **stripping non-content** (nav, footer, script, style) before
extracting text.

Watch what happens if you DON'T strip — the retrieval-relevant content gets
buried in menu items and copyright notices.

In [5]:
%pip install BeautifulSoup4

Note: you may need to restart the kernel to use updated packages.


In [6]:
from bs4 import BeautifulSoup

html = SAMPLE_HTML.read_text()

# Naive: get all text, no stripping
soup_naive = BeautifulSoup(html, "html.parser")
naive_text = soup_naive.get_text(separator="\n", strip=True)
print("══ NAIVE extraction (nav + footer + scripts included) ══")
print(naive_text[:400])
print(f"\n... {len(naive_text)} chars total\n")

══ NAIVE extraction (nav + footer + scripts included) ══
kinkyShoeBros — Product Selection & Fitment Guide
kinkyShoeBros Fitment Hub & Shoe Catalog
Finding the right pair is about getting the perfect fit for your unique foot shape. Use our 
        comprehensive fitment table below to match our top 16 best-selling models to your foot profile.
Customer Fitment Matrix (15+ Models)
Brand
Model
Best For (Foot Type)
Fitment Note
Category
Nike
Air Max 90
Neut

... 1518 chars total



In [7]:
# Better: strip non-content tags first
soup_clean = BeautifulSoup(html, "html.parser")
for tag in soup_clean(["nav", "footer", "script", "style"]):
    tag.decompose()

# Get <main> if present, else body
main = soup_clean.find("main") or soup_clean.find("body") or soup_clean
clean_text = main.get_text(separator="\n", strip=True)

print("══ CLEAN extraction (nav + footer + scripts stripped) ══")
print(clean_text[:400])
print(f"\n... {len(clean_text)} chars total")
print(f"\nRemoved: {len(naive_text) - len(clean_text)} chars of noise")

══ CLEAN extraction (nav + footer + scripts stripped) ══
kinkyShoeBros Fitment Hub & Shoe Catalog
Finding the right pair is about getting the perfect fit for your unique foot shape. Use our 
        comprehensive fitment table below to match our top 16 best-selling models to your foot profile.
Customer Fitment Matrix (15+ Models)
Brand
Model
Best For (Foot Type)
Fitment Note
Category
Nike
Air Max 90
Neutral / Average
True to size (TTS)
Lifestyle
Nike
Ai

... 1468 chars total

Removed: 50 chars of noise


**Discussion:** the naive extraction leaks 'Home | Products | Pricing | Docs'
into the top of your chunk. Retrieval then matches that noise against user
queries about 'home page' or 'pricing' — retrieving your product page for
questions it shouldn't answer.

**Rule of thumb for HTML:** always strip `<nav>`, `<footer>`, `<script>`,
`<style>` before extracting text. Add `<aside>` if your source has sidebars.

---

## Cell 5 — python-docx on the DOCX

**python-docx** parses DOCX files (Word documents). Each paragraph has a
`style` — 'Title', 'Heading 1', 'Heading 2', 'Normal', etc. These styles are
**gold** for structure-aware chunking.

In [8]:
%pip install python-docx

Note: you may need to restart the kernel to use updated packages.


In [9]:
from docx import Document

doc = Document(str(SAMPLE_DOCX))

print(f"DOCX: {SAMPLE_DOCX.name}")
print(f"Paragraphs: {len(doc.paragraphs)}\n")

for i, para in enumerate(doc.paragraphs):
    if para.text.strip():
        style = para.style.name
        preview = para.text[:70]
        print(f"  [{i:2d}] {style:15s}  {preview}...")

DOCX: kinkyshoebros_preloved.docx
Paragraphs: 5

  [ 0] Title            kinkyShoeBros Pre-Loved Shoe Sales Program...
  [ 1] Normal           The kinkyShoeBros Pre-Loved Program extends the lifecycle of premium f...
  [ 2] Heading 1        Trade-In Grading & Valuation Table...
  [ 3] Heading 2        Support Contacts...
  [ 4] Normal           For verification disputes, contact the Pre-Loved Support desk at prelo...


**Notice the `style` column.** Heading 1 / Heading 2 tell us where sections
start. That structure is exactly what we'll use in structure-aware chunking.

PyMuPDF and BeautifulSoup don't give you clean structure like this by default —
DOCX is the friendliest format for ingestion.

---

## Cell 6 — The scanned PDF trap

**Some PDFs contain no text — only images of pages.** These come from scanned
paper documents. PyMuPDF returns empty strings for such pages. Your ingestion
pipeline appears to work but produces zero chunks.

Let's simulate this: create a PDF with just an image (no text layer) and see
what happens.

In [10]:
# Create a fake 'scanned' PDF — one page with just a filled rectangle, no text
import pymupdf

scanned_path = SAMPLE_DIR / "scanned_fake.pdf"
scan_doc = pymupdf.open()
page = scan_doc.new_page()
# Draw a filled rectangle to simulate a scanned image with no text layer
page.draw_rect(pymupdf.Rect(100, 100, 500, 700), fill=(0.9, 0.9, 0.9))
scan_doc.save(str(scanned_path))
scan_doc.close()

# Now try to extract text
doc = pymupdf.open(scanned_path)
extracted = doc[0].get_text()
doc.close()

print(f"Fake 'scanned' PDF: {scanned_path.name}")
print(f"Extracted text length: {len(extracted)} chars")
print(f"Extracted content: {extracted!r}")
print()
print("That's the trap. The parser succeeded — no crash. But zero content.")
print("Your ingestion silently produced 0 chunks from this document.")

# Cleanup — don't clutter sample_docs/
scanned_path.unlink()

Fake 'scanned' PDF: scanned_fake.pdf
Extracted text length: 0 chars
Extracted content: ''

That's the trap. The parser succeeded — no crash. But zero content.
Your ingestion silently produced 0 chunks from this document.


**Fix:** run OCR on scanned PDFs before parsing.
Popular options: `pytesseract`, `easyocr`, or cloud services (AWS Textract,
Google Document AI).

**For today:** add a check to your ingestion pipeline: if a PDF parses to
< 50 chars of text, log a warning. That'll catch the silent-failure case.

---

## Cell 7 — Parsing Recap

**Four libraries, one decision matrix:**

| Format | Library | Speed | Structure | Tables |
|---|---|---|---|---|
| PDF (prose) | PyMuPDF | fast | pages only | poor |
| PDF (tables) | pdfplumber | slower | pages only | good |
| HTML | BeautifulSoup | fast | tags | table tags |
| DOCX | python-docx | fast | styles (best!) | table objects |

**Traps to remember:**
- HTML: don't include `<nav>` and `<footer>` — they leak into your chunks
- PDF: some PDFs are scanned images → empty text → silent failure
- All formats: check output length before assuming parse succeeded

Now to Topic 2 — chunking.

---

## Cell 8 — Chunking strategy #1: Fixed-size (Baseline)

Fixed-size chunking uses a sliding window. Simple, but cuts through
sentences and paragraphs indiscriminately.

Let's run it on the PDF's page 1 text and see the damage.

In [11]:
# Load the PDF text (page 1 only, for demo compactness)
doc = pymupdf.open(SAMPLE_PDF)
page1_text = doc[0].get_text()
doc.close()

def chunk_fixed(text, size=400, overlap=40):
    """Baseline sliding-window chunker."""
    if len(text) <= size:
        return [text]
    chunks, i = [], 0
    while i < len(text):
        end = min(i + size, len(text))
        chunks.append(text[i:end])
        if end == len(text):
            break
        i = end - overlap
    return chunks

fixed_chunks = chunk_fixed(page1_text, size=400, overlap=40)
print(f"Fixed-size chunking (size=400, overlap=40) → {len(fixed_chunks)} chunks\n")

for i, chunk in enumerate(fixed_chunks):
    ends_mid = not chunk.rstrip().endswith(('.', '!', '?', ':'))
    marker = " ← ends mid-sentence" if ends_mid else ""
    preview = chunk[-60:].replace('\n', ' ')
    print(f"  [{i}] {len(chunk):>4d} chars, ends: '...{preview}'{marker}")

Fixed-size chunking (size=400, overlap=40) → 4 chunks

  [0]  400 chars, ends: '...as, and New Balance releases. Approved Employee Allocation R' ← ends mid-sentence
  [1]  400 chars, ends: '...evel 1 Nike Air Force 1 Annually Stockroom Manager PUMA Sued' ← ends mid-sentence
  [2]  400 chars, ends: '...rly HR Business Partner New Balance 550 Bi-Annually Finance ' ← ends mid-sentence
  [3]  158 chars, ends: '... Annually IT Support Specialist Nike Pegasus 40 Bi-Annually ' ← ends mid-sentence


**Discussion:** count the mid-sentence cuts. Each one is a chunk that starts
with fragment-context — bad for retrieval. This is why we need better chunking.

---

## Cell 9 — Chunking strategy #2: Recursive

**Recursive chunking:** try paragraphs first; if a paragraph is too long, fall
back to sentence splits within it. Preserves boundaries whenever possible.

This is the strategy most systems default to.

In [12]:
import re

def chunk_recursive(text, max_size=400):
    """Split by paragraphs. If a paragraph exceeds max_size, split by sentences."""
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    chunks = []
    for para in paragraphs:
        if len(para) <= max_size:
            chunks.append(para)
        else:
            # Sentence-level fallback
            sentences = re.split(r'(?<=[.!?])\s+', para)
            current = ""
            for sent in sentences:
                if len(current) + len(sent) + 1 <= max_size:
                    current = (current + " " + sent).strip()
                else:
                    if current:
                        chunks.append(current)
                    current = sent
            if current:
                chunks.append(current)
    return chunks

recursive_chunks = chunk_recursive(page1_text, max_size=400)
print(f"Recursive chunking (max_size=400) → {len(recursive_chunks)} chunks\n")

for i, chunk in enumerate(recursive_chunks):
    ends_mid = not chunk.rstrip().endswith(('.', '!', '?', ':'))
    marker = " ← ends mid-sentence" if ends_mid else " ← clean end"
    preview = chunk[-60:].replace('\n', ' ')
    print(f"  [{i}] {len(chunk):>4d} chars, ends: '...{preview}'{marker}")

Recursive chunking (max_size=400) → 2 chunks

  [0]  369 chars, ends: '...fitted in the latest Nike, adidas, and New Balance releases.' ← clean end
  [1]  867 chars, ends: '...X Annually IT Support Specialist Nike Pegasus 40 Bi-Annually' ← ends mid-sentence


**Compare with Fixed chunking:**
- Fixed had multiple mid-sentence cuts → messy chunks
- Recursive respects paragraph AND sentence boundaries → cleaner chunks
- Chunk sizes are variable (that's the trade-off), but every chunk is
  semantically self-contained

**Recursive is the sane default for most RAG systems.** Use it unless you have
a specific reason to try something else.

---

## Cell 10 — Chunking strategy #3: Semantic (simplified demo)

**Semantic chunking:** embed each sentence, group consecutive sentences whose
cosine similarity is above a threshold. Similar sentences stay together;
topic shifts trigger new chunks.

**Simplified demo:** hand-pick 6 sentences covering 2 topics from KinkyShoeBros
(Employee Allocation vs Pre-Loved Grading), embed them, and show the groupings.

In [13]:
%pip install -q ollama

import ollama

OLLAMA_CHAT_MODEL = "gpt-oss:20b"
OLLAMA_EMBED_MODEL = "nomic-embed-text"

_USE_NOMIC_PREFIXES = OLLAMA_EMBED_MODEL.startswith("nomic-embed-text")
EMBED_QUERY_PREFIX = "search_query: " if _USE_NOMIC_PREFIXES else ""
EMBED_DOC_PREFIX = "search_document: " if _USE_NOMIC_PREFIXES else ""

def _is_pulled(wanted: str, installed: list[str]) -> bool:
    return any(n == wanted or (":" not in wanted and n.split(":")[0] == wanted) for n in installed)

def check_ollama_ready() -> bool:
    try:
        listed = ollama.list()
    except Exception as err:
        raise ConnectionError(f"Cannot reach the Ollama server ({err}).") from err
    installed = [getattr(m, "model", None) or m["model"] for m in listed.models]
    missing = [m for m in (OLLAMA_CHAT_MODEL, OLLAMA_EMBED_MODEL) if not _is_pulled(m, installed)]
    if missing:
        raise RuntimeError("Missing Ollama model(s): " + ", ".join(missing))
    return True

EMBED_DIM = None
try:
    check_ollama_ready()
    response = ollama.chat(
        model=OLLAMA_CHAT_MODEL,
        messages=[{"role": "user", "content": "What is the core benefit of RAG? Answer with exactly one word."}]
    )
    print(f"Sample reply from {OLLAMA_CHAT_MODEL}: {response.message.content}")
    EMBED_DIM = len(ollama.embed(model=OLLAMA_EMBED_MODEL, input=["dimension probe"]).embeddings[0])
    print(f"Embedding model '{OLLAMA_EMBED_MODEL}' -> {EMBED_DIM} dimensions.")
except (ConnectionError, RuntimeError, ollama.ResponseError) as err:
    EMBED_DIM = 768
    print(f"[HANDLED NEGATIVE SCENARIO] Ollama setup problem: {err}")


Note: you may need to restart the kernel to use updated packages.
Sample reply from gpt-oss:20b: recall
Embedding model 'nomic-embed-text' -> 768 dimensions.


In [14]:
import os
import numpy as np
import ollama

# 6 sentences, 2 topics — Employee Allocation + Preloved Grading
sentences = [
    "Retail managers are allocated Nike Air Max 95s quarterly.",
    "Sales associates receive New Balance or ASICS twice a year.",
    "The company provides a quarterly stipend for all footwear.",
    "Grade 10 deadstock shoes yield 85% trade-in payout.",
    "Grade 8 shoes have light creasing and clean insoles.",
    "Grade 4 shoes are beater condition with heavy scuffs.",
]

prefixed_sentences = [f"{EMBED_DOC_PREFIX}{s}" for s in sentences]

resp = ollama.embed(
    model=OLLAMA_EMBED_MODEL,
    input=prefixed_sentences
)
embeddings = [np.array(emb) for emb in resp.embeddings]

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print("Consecutive sentence similarities:\n")
for i in range(len(sentences) - 1):
    sim = cosine(embeddings[i], embeddings[i+1])
    print(f"  s{i} → s{i+1}: cosine = {sim:.3f}")
    print(f"    s{i}: {sentences[i]}")
    print(f"    s{i+1}: {sentences[i+1]}\n")

Consecutive sentence similarities:

  s0 → s1: cosine = 0.728
    s0: Retail managers are allocated Nike Air Max 95s quarterly.
    s1: Sales associates receive New Balance or ASICS twice a year.

  s1 → s2: cosine = 0.724
    s1: Sales associates receive New Balance or ASICS twice a year.
    s2: The company provides a quarterly stipend for all footwear.

  s2 → s3: cosine = 0.678
    s2: The company provides a quarterly stipend for all footwear.
    s3: Grade 10 deadstock shoes yield 85% trade-in payout.

  s3 → s4: cosine = 0.673
    s3: Grade 10 deadstock shoes yield 85% trade-in payout.
    s4: Grade 8 shoes have light creasing and clean insoles.

  s4 → s5: cosine = 0.804
    s4: Grade 8 shoes have light creasing and clean insoles.
    s5: Grade 4 shoes are beater condition with heavy scuffs.



In [15]:
# Now group consecutive sentences into chunks — split whenever similarity drops below threshold
THRESHOLD = 0.55  # tuned for this demo; real systems calibrate per corpus

chunks = [[sentences[0]]]
for i in range(1, len(sentences)):
    sim = cosine(embeddings[i-1], embeddings[i])
    if sim >= THRESHOLD:
        chunks[-1].append(sentences[i])
    else:
        chunks.append([sentences[i]])  # new chunk starts here

print(f"Semantic chunking (threshold={THRESHOLD}) → {len(chunks)} chunks\n")
for i, chunk_sentences in enumerate(chunks):
    print(f"══ Chunk {i} ({len(chunk_sentences)} sentences) ══")
    for s in chunk_sentences:
        print(f"  • {s}")
    print()

Semantic chunking (threshold=0.55) → 1 chunks

══ Chunk 0 (6 sentences) ══
  • Retail managers are allocated Nike Air Max 95s quarterly.
  • Sales associates receive New Balance or ASICS twice a year.
  • The company provides a quarterly stipend for all footwear.
  • Grade 10 deadstock shoes yield 85% trade-in payout.
  • Grade 8 shoes have light creasing and clean insoles.
  • Grade 4 shoes are beater condition with heavy scuffs.



**Discussion:**
- Did the chunker group the 3 allocation sentences together and separate them
  from the grading sentences? Look at the similarity between s2 and s3.
- This works when your corpus has natural topic shifts. It's overkill for
  short structured documents.
- **When to use:** documents without clear headings but with clear topic shifts
  (blog posts, articles, transcripts).

---

## Cell 11 — Chunking strategy #4: Structure-aware

**Structure-aware chunking:** use the document's OWN structure as split points.
For DOCX, use heading styles. For HTML, use `<h1>` / `<h2>`. For Markdown, use
`#` / `##`.

This is the **best strategy when your documents have real structure** — which
the DOCX file does.

In [16]:
from docx import Document

def chunk_docx_structure_aware(path):
    """Split a DOCX into chunks bounded by Heading 1 / Heading 2 styles.
    Attach section_path metadata based on the heading hierarchy."""
    doc = Document(str(path))
    
    chunks = []
    current_h1 = None
    current_h2 = None
    current_text = []
    
    def flush():
        if current_text:
            section_path = " > ".join(p for p in [current_h1, current_h2] if p)
            chunks.append({
                "section_path": section_path,
                "text": "\n".join(current_text),
            })
    
    for para in doc.paragraphs:
        if not para.text.strip():
            continue
        style = para.style.name
        
        if style == "Title" or style == "Heading 1":
            flush()  # save what we have
            current_text = []
            current_h1 = para.text
            current_h2 = None
        elif style == "Heading 2":
            flush()  # save subsection
            current_text = []
            current_h2 = para.text
        else:
            current_text.append(para.text)
    
    flush()  # final flush
    return chunks

structured = chunk_docx_structure_aware(SAMPLE_DOCX)
print(f"Structure-aware chunking of {SAMPLE_DOCX.name} → {len(structured)} chunks\n")

for i, chunk in enumerate(structured):
    print(f"══ Chunk {i} ══")
    print(f"  section_path: {chunk['section_path']!r}")
    preview = chunk['text'][:120].replace('\n', ' ')
    print(f"  text:         {preview}...")
    print(f"  length:       {len(chunk['text'])} chars\n")

Structure-aware chunking of kinkyshoebros_preloved.docx → 2 chunks

══ Chunk 0 ══
  section_path: 'kinkyShoeBros Pre-Loved Shoe Sales Program'
  text:         The kinkyShoeBros Pre-Loved Program extends the lifecycle of premium footwear. Every trade-in is evaluated against our 1...
  length:       217 chars

══ Chunk 1 ══
  section_path: 'Trade-In Grading & Valuation Table > Support Contacts'
  text:         For verification disputes, contact the Pre-Loved Support desk at preloved@kinkyshoebros.com. Program Lead: Priya Sharma ...
  length:       132 chars



**Look at that.** Each chunk has:
- A `section_path` like `'Trade-In Grading & Valuation Table > Support Contacts'`
- Text that is exactly the content under that heading — no more, no less

**Why this is powerful:**
1. **Cleaner retrieval** — a chunk about 'Support Contacts' won't drift into 'Grading' content
2. **Metadata comes for free** — `section_path` is already extracted, ready for a metadata schema
3. **Explainable citations** — 'this answer came from section: Pre-loved > Support Contacts'

**Rule of thumb:** if your document has headings, use structure-aware chunking.
It's the biggest single quality lift you can make to a RAG pipeline.

---

## Cell 12 — Wrap & Decision Matrix

**In this notebook you:**
1. Parsed 3 real document formats — PyMuPDF, pdfplumber, BeautifulSoup, python-docx
2. Saw the scanned-PDF trap (empty text → silent failure)
3. Compared 4 chunking strategies on real content — fixed, recursive, semantic, structure-aware
4. Confirmed structure-aware is the biggest quality lift when documents have headings

**Decision matrix for your projects:**

| Your corpus looks like… | Chunking strategy |
|---|---|
| Docs with clear headings (policies, manuals, guides) | Structure-aware |
| Narrative text with topic shifts (blog posts, articles) | Semantic |
| Mixed prose without structure | Recursive |
| Very short / uniform docs | Fixed |
